# 08 - Train a PPO Model Online

This notebook shows on-policy PPO training in Mouse Core. It mirrors the online DQN loop in `03_train_online.ipynb`, but swaps the action-value head and TD loss for an actor-critic pair:

1. Step a `GroupEnv` and sample actions from the stochastic policy (`predictions["action"]`).
2. Store each transition with its behavior `old_log_prob` in an on-policy `Datastore`.
3. After the rollout, sample those rows with `DataLoader` and train with `PpoObjective` (GAE + clipped surrogate + value + entropy).
4. Clear the on-policy stores and repeat until the optimizer budget is reached.

The shared Mouse Core pieces stay the same: row dicts, `Datastore` / `DataLoader`, `Model.forward`, and a plain objective callable. Only the heads, action sampling, and objective change.


In [ ]:
from collections import deque

import torch

import procedural_frozenlake  # noqa: F401 — registers Procedural-FrozenLake-v1
from mouse_gym import EnvConfig, make_group_env

from mouse_core.data import DataLoader, Datastore
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import NumericEmbedder
from mouse_core.models.heads import DiscreteActionHead, SwiGLUHead
from mouse_core.objectives import PpoObjective, batch_field, sample_discrete_action


MODEL_ID = "mouse-example-model-ppo"  # Hugging Face model repo for push_model_to_hub
MAX_ACTIONS = 4  # number of discrete actions predicted by the policy head
MAX_OBS_DISCRETE = 64  # vocabulary size for discrete observations
MAX_EPISODE_STEPS = 30  # environment-specific episode limit
EPISODES_PER_TASK = 20  # environment-specific task length
NUM_ENVS = 30  # number of environment streams in the GroupEnv

GRADIENT_STEPS = 20000  # total optimizer updates
SEQUENCE_LENGTH = 512  # sequence length sampled by DataLoader
BATCH_SIZE = 4  # sequences per optimizer step

STEPS_PER_ROLLOUT = 500  # new rows collected per env before training
PPO_EPOCHS = 4  # passes over the on-policy buffer after each rollout
GRADIENT_STEPS_PER_EPOCH = 50  # optimizer updates per PPO epoch
# Per rollout: PPO_EPOCHS * GRADIENT_STEPS_PER_EPOCH = 200 updates (same as online DQN)

LEARNING_STARTS = 15_000  # env steps collected before the first optimizer update


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Environment

Online PPO uses the same `EnvConfig` / `make_group_env` setup as the other live-env notebooks. Keep row fields consistent with the model modalities and with `PpoObjective` (`action`, `observation`, `reward`, `done`, plus rollout-only `old_log_prob`).


In [ ]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_ppo_{i}",
        reset_seed=i,
        episodes_per_task=EPISODES_PER_TASK,
        task_reset_options={"regenerate_map": True},  # forwarded to the environment at task reset
        kwargs={
            "width": 8,
            "height": 8,
            "max_episode_steps": MAX_EPISODE_STEPS,
            "map_seed": i,
            "slippery_success_rate": 1.0,  # environment-specific option
            "permute_obs": True,      # environment-specific option
            "permute_actions": True,  # environment-specific option
        },
    )
    for i in range(NUM_ENVS)
]

env = make_group_env(configs)


## Build The Model

PPO needs two heads on a shared backbone:

* `DiscreteActionHead` → `predictions["action"]` policy logits (also `action_head` for inference)
* `SwiGLUHead(out_features=1)` → `predictions["value"]` scalar V(s)

`old_log_prob` is **not** an encoder modality — it is stored on rollout rows and injected into `objective_data` with `batch_field` before the objective call.


In [ ]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = NumericEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "in_min": 0.01,
            "in_max": 100.0,
            "tokens": 1,
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1,
        },
    ],
    modality_fusion="sum",
    include_type_token=False,
)

policy_head = DiscreteActionHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)
value_head = SwiGLUHead(
    in_features=backbone.hidden_dim,
    out_features=1,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(
    encoder=encoder,
    backbone=backbone,
    heads={"action": policy_head, "value": value_head},
    action_head="action",
).train().to(device)
model.backbone.model.compile()  # optional compile step for faster repeated forwards
print(model)


## On-Policy Buffer And Contexts

Each env writes to one on-policy `Datastore` that is cleared after every training phase. Bounded `contexts` deques keep recent history for cached rollouts (in-context conditioning), separate from the on-policy buffer lifetime.


In [ ]:
stores = [Datastore(name=name) for name in env.names]
contexts = [deque(maxlen=SEQUENCE_LENGTH) for _ in env.names]


## Rollout Phase

Unlike ε-greedy DQN, PPO explores by sampling from the current categorical policy. For each lockstep step:

1. Run `model(..., use_cache=True)` on pending context rows.
2. `sample_discrete_action` draws actions and behavior log-probs from `predictions["action"]`.
3. Step the `GroupEnv` and append flat rows that include `old_log_prob`.

`old_log_prob` sits on the same step as `action` (the transition that produced the next observation), matching the timing used by `PpoObjective`.


In [ ]:
def rollout(
    model: Model,
    env,
    stores: list[Datastore],
    contexts: list[deque],
    env_steps: int,
    grad_steps: int,
) -> int:
    """Collect on-policy rows (with behavior log-probs) into datastores."""
    env.metrics.clear()
    model.eval()

    kv_cache = None
    pending_rows = [list(c) for c in contexts]

    for step in range(STEPS_PER_ROLLOUT):
        # Need at least one context row before the policy can act.
        ready = [bool(c) for c in contexts]
        log_probs = [0.0] * len(contexts)
        if any(ready):
            with torch.no_grad():
                predictions, _, kv_cache = model(pending_rows, cache=kv_cache, use_cache=True)
            logits = predictions["action"][:, -1]
            actions_t, log_probs_t = sample_discrete_action(logits, num_actions=MAX_ACTIONS)
            pending_rows = [[] for _ in contexts]
        random_inputs = env.sample_random_input()
        inputs = []
        for i in range(len(contexts)):
            if ready[i]:
                inputs.append({"action": actions_t[i].cpu().numpy()})
                log_probs[i] = float(log_probs_t[i].item())
            else:
                # First step of an empty stream: random action, dummy log-prob.
                inputs.append(random_inputs[i])
                log_probs[i] = 0.0
        outputs = env.step(inputs)
        for i, out in enumerate(outputs):
            row = {**inputs[i], **out, "old_log_prob": log_probs[i]}
            row.pop("info", None)  # keep stored rows flat
            stores[i].append(row)
            contexts[i].append(row)
            pending_rows[i].append(row)
        env_steps += len(outputs)
        if (step + 1) % 100 == 0:
            task_scores = [r for env_tasks in env.metrics.task_cum_rewards for r in env_tasks]
            mean_task_score = sum(task_scores) / len(task_scores) if task_scores else float("nan")
            print(
                f"  rollout step {step + 1}/{STEPS_PER_ROLLOUT} | env_step={env_steps} grad_step={grad_steps}"
                f" | {len(task_scores)} tasks"
                f" | mean task score {mean_task_score:.2f}/{EPISODES_PER_TASK}"
            )
            env.metrics.clear()

    del kv_cache
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return env_steps


## Training Phase

After a rollout, `loader.refresh()` rescans the on-policy stores. Each PPO epoch runs a fixed number of optimizer updates. Before `PpoObjective`, `batch_field` copies `old_log_prob` from the sampled rows into `objective_data` (it is not an encoder modality).

Discounts match the task structure used by the DQN notebooks: undiscounted within a task, hard cut at task boundaries.


In [ ]:
loader = DataLoader(
    stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    weight_mode="per_step",
    pack=True,
    num_workers=0,
)
objective = PpoObjective(
    gamma_step=1.0,
    gamma_episode_terminal=1.0,
    gamma_episode_truncated=1.0,
    gamma_task_terminal=0.0,
    gamma_task_truncated=0.0,
    gae_lambda=0.95,
    clip_eps=0.2,
    vf_coef=0.5,
    ent_coef=0.01,
    num_actions=MAX_ACTIONS,
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8,
)


In [ ]:
def train(
    model: Model,
    optimizer: torch.optim.Optimizer,
    objective: PpoObjective,
    loader: DataLoader,
    env_steps: int,
    grad_steps: int,
    num_updates: int,
) -> tuple[torch.Tensor, dict[str, float]]:
    """Refresh the on-policy buffer and run ``num_updates`` PPO optimizer steps."""
    model.train()
    loader.refresh()

    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for update in range(num_updates):
        batch, segment_ids = loader.next_batch()

        predictions, objective_data, _ = model(batch, segment_ids=segment_ids)
        objective_data["old_log_prob"] = batch_field(
            batch, "old_log_prob", device=objective_data.device
        )
        loss, metrics = objective(objective_data, predictions)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if (update + 1) % 50 == 0 or update + 1 == num_updates:
            print(
                f"env_step={env_steps} grad_step={grad_steps + update + 1}"
                f" | loss={loss.item():.4f}"
                f" policy={metrics['policy_loss']:.4f}"
                f" value={metrics['value_loss']:.4f}"
                f" ent={metrics['entropy']:.4f}"
                f" kl={metrics['approx_kl']:.4f}"
            )

    assert loss is not None
    return loss, metrics


## Run Online PPO

Each cycle collects a fresh on-policy rollout, runs `PPO_EPOCHS` passes of updates, then clears the buffers so the next cycle does not reuse stale behavior log-probs.


In [ ]:
env_steps = 0
grad_steps = 0

while grad_steps < GRADIENT_STEPS:
    env_steps = rollout(model, env, stores, contexts, env_steps, grad_steps)

    if env_steps >= LEARNING_STARTS:
        for _epoch in range(PPO_EPOCHS):
            num_updates = min(GRADIENT_STEPS_PER_EPOCH, GRADIENT_STEPS - grad_steps)
            if num_updates <= 0:
                break
            train(model, optimizer, objective, loader, env_steps, grad_steps, num_updates)
            grad_steps += num_updates
        for store in stores:
            store.clear()
        if device.type == "cuda":
            torch.cuda.empty_cache()

loader.close()
env.close()
print(f"Online PPO finished ({grad_steps} optimizer steps, {env_steps} env steps).")


## Push To The Hub

Run this cell to save the PPO checkpoint. The inference notebook can load it with `load_model` / the Hub model id used here.


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")
